In [1]:
!pip install --upgrade google-meridian[colab,and-cuda,schema]

import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
from itertools import permutations
from google.colab import drive
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp
import sys
import os

# Mount a storage
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

!git clone --branch meridian_modeling https://github.com/pstat197/BlueAlpha3-Synergy-Analysis

df = pd.read_csv("/content/BlueAlpha3-Synergy-Analysis/data/monthly_mocha.csv")
df = df.loc[:, (df != 0).any()]

import sys
sys.path.append("/content/BlueAlpha3-Synergy-Analysis/scripts")
from geometric_mean import create_geometric_mean_interactions

df_mmm = create_geometric_mean_interactions(df)

print(df_mmm.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3

In [2]:
import sys
sys.path.append("/content/BlueAlpha3-Synergy-Analysis/scripts")
from geometric_mean import create_geometric_mean_interactions

df_mmm = create_geometric_mean_interactions(df)

print(df_mmm.head())
# Separate spend and impressions columns
df_mmm = df_mmm.rename(columns={'date': 'time'}).sort_values(by='time', ascending=False).reset_index(drop=True)

# Convert the 'time' column to datetime objects with the specified format
df_mmm['time'] = pd.to_datetime(df_mmm['time'], format='%m/%d/%y')

spend_cols = [c for c in df_mmm.columns if c.endswith("_spend")]
impressions_cols = [c for c in df_mmm.columns if c.endswith("_impressions")]

channels = sorted({
    c.replace("_spend", "").replace("_impressions", "")
    for c in (spend_cols + impressions_cols)
})

assert len(channels) == len(spend_cols) == len(impressions_cols)

df_mmm['revenue_kpi'] = df_mmm['subscriptions'] * 100

kpi_col = "revenue_kpi"

builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="revenue",
    default_kpi_column=kpi_col
)

builder = (
    builder.with_kpi(
        df_mmm
    )
    .with_media(
        df_mmm,
        media_cols=spend_cols,
        media_spend_cols=spend_cols,
        media_channels=channels
    )
)

mmm_data = builder.build()

      date  subscriptions   meta_spend  meta_impressions  google_spend  \
0   8/4/25          15540  91538.06648          16572258   116667.9945   
1  7/28/25          14525  93840.18612          25300600   180486.9558   
2  7/21/25          16880  48403.06780          14099214   200817.3250   
3  7/14/25          20113  49470.96783          13652072   215770.9242   
4   7/7/25          16492  48948.28744          10121002   209231.9668   

   google_impressions  snapchat_spend  snapchat_impressions  tiktok_spend  \
0             6473132     94750.04035               3420454           0.0   
1             9487127     99447.23218               3235285           0.0   
2             7909118     84738.57435               4766750           0.0   
3             7789279     83204.40500               4022680           0.0   
4             6806878     82642.37271               4532105           0.0   

   tiktok_impressions  ...  liveintent_spend  liveintent_impressions  \
0                   

/usr/local/lib/python3.12/dist-packages/meridian/data/input_data.py:517: UserWarning: Revenue from the `kpi` data is used when `kpi_type`=`revenue`. `revenue_per_kpi` is ignored.
  warnings.warn(


In [6]:
# Initialize model
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=0.2,
        scale=0.9,
        name=constants.ROI_M
    )
)

# Prepare holdout id for testing data
n_geos = 1
n_times = len(df_mmm.time)

np.random.seed(42)
test_pct = 0.2
num_holdout = int(n_times * test_pct)

holdout_id = np.full((n_geos, n_times), False)

holdout_indices = np.random.choice(n_times, size=num_holdout, replace=False)
holdout_id[0, holdout_indices] = True
holdout_id = holdout_id.flatten()

print(f"Created holdout_id with shape: {holdout_id.shape}")
print(f"Number of holdout periods: {np.sum(holdout_id)}")

model_spec = spec.ModelSpec(
    prior=prior,
    enable_aks=True,
    holdout_id=holdout_id,
    media_prior_type='roi',
    non_media_treatments_prior_type='contribution'
)

mmm = model.Meridian(
    input_data=mmm_data,
    model_spec=model_spec
)

# Sample from the model
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

Created holdout_id with shape: (74,)
Number of holdout periods: 14


/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:74: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-pa

In [7]:
# Model diagnostics
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

/tmp/ipykernel_11583/1500263722.py:2: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Metric check,Status,Recommended action
Convergence,Pass,"The model has likely converged, as all parameters have R-hat values < 1.2."
Baseline,Pass,The posterior probability that the baseline is negative is 0.00. We recommend visually inspecting the baseline time series in the Model Fit charts to confirm this.
Bayesian p-value,Pass,The Bayesian posterior predictive p-value is 0.84. The observed total outcome is consistent with the model's posterior predictive distribution.
Goodness of fit,Pass,"R-squared = 0.8904 (All), 0.9047 (Train), 0.8427 (Test); MAPE = 0.0491 (All), 0.0423 (Train), 0.0780 (Test); wMAPE = 0.0489 (All), 0.0430 (Train), 0.0741 (Test). These goodness-of-fit metrics are intended for guidance and relative comparison."
Prior-posterior shift,Pass 10/10 channels passed,The model has successfully learned from the data. This is a positive sign that your data was informative.


In [8]:
# Two-page summary
mmm_summarizer = summarizer.Summarizer(mmm)

filepath = meridian_root
start_date = str(df_mmm["time"].min().date()) # Use df_lagged for start_date
end_date = str(df_mmm["time"].max().date()) # Use df_lagged for end_date
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:3356: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:1033: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.warn(


Dataset,R-squared,MAPE,wMAPE
Training Data,0.90,4%,4%
Testing Data,0.84,8%,7%
All Data,0.89,5%,5%


In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

CPU times: user 3min 29s, sys: 2.2 s, total: 3min 31s
Wall time: 2min 38s


In [ ]:
filepath = meridian_root
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/optimization_output.html')

Channel,Non-optimized spend,Optimized spend
amazon_x_meta,47%,44%
beehiiv,20%,17%
tiktok,12%,14%
google,8%,9%
google_x_liveintent,4%,5%
liveintent,4%,5%
amazon,4%,4%
meta,2%,2%
moloco,0%,0%
snapchat,0%,0%
